# Delta Live Tables (DLT) — Declarative ETL
### Databricks Data Engineer Associate — ILT Teaching Notebook

---

## Exam Weightage

| Section | Weight | Questions |
|---------|--------|-----------|
| Databricks Lakehouse Platform | 24% | ~11 |
| ETL with Spark SQL and Python | 29% | ~13 ← HIGHEST |
| Incremental Data / Streaming | 20% | ~9 |
| **Production Pipelines (DLT)** | **16%** | **~7 ← THIS SESSION** |
| Data Governance | 9% | ~5 |

> **Strategy:** DLT = ~7 exam questions. Understand declarative concept + all 3 dataset types + data quality + pipeline modes.

---
## What we cover today
1. Procedural vs Declarative ETL — why DLT was created
2. DLT Dataset Types — Streaming Table, Materialized View, View
3. SQL Syntax — CREATE OR REFRESH, LIVE, read_files()
4. Python Syntax — @dlt.table, @dlt.view, dlt.read()
5. Data Quality Expectations — Warn, Drop, Fail
6. Pipeline Modes × Environments — 4 combinations
7. Certification exam tips throughout

---
## Chapter 1 — Procedural vs Declarative ETL

### The Problem With Procedural ETL

```
Traditional (Procedural) Approach — "HOW" you build it:

  Step 1: Read raw CSV files
  Step 2: Define schema manually
  Step 3: Add auto loader
  Step 4: Handle schema evolution
  Step 5: Add checkpointing for fault tolerance
  Step 6: Write to Bronze
  Step 7: Apply transformations
  Step 8: Handle MERGE (upserts) for Silver
  Step 9: Write to Silver
  Step 10: Add data quality checks manually
  Step 11: Build aggregations
  Step 12: Write to Gold
  Step 13: Monitor and orchestrate everything
  ...hundreds of lines of code
```

### The DLT Solution

```
DLT (Declarative) Approach — "WHAT" you want:

  @dlt.table
  def bronze_orders():
      return spark.readStream.format("cloudFiles")...

  @dlt.table
  @dlt.expect("valid amount", "amount > 0")
  def silver_orders():
      return dlt.read("bronze_orders").filter(...)

  @dlt.table
  def gold_sales():
      return dlt.read("silver_orders").groupBy("channel").agg(...)

  You write WHAT the output should be.
  DLT handles fault tolerance, checkpointing, orchestration, monitoring.
```

### Analogy — Building a House

```
Procedural ETL:  Architect tells workers:
  "Lay 100 bricks at position X,Y. Mix cement ratio 1:3. Wait 2 hours. Repeat."

DLT:  Architect gives blueprint:
  "Build 4-bedroom house, 2 bathrooms, kitchen on ground floor."
  Workers figure out HOW to build it.

DLT = you describe the RESULT. Databricks figures out the execution.
```

### Key Capability Comparison

| Feature | Procedural ETL | Delta Live Tables |
|---------|---------------|------------------|
| Fault tolerance | You write checkpointing code | Automatic |
| Schema evolution | You handle manually | Automatic |
| Data quality | You write validation code | Built-in EXPECT constraints |
| Orchestration | External (Airflow, Jobs) | Built-in |
| Lineage tracking | Not available | Built-in visual DAG |
| Error recovery | Manual retry logic | Automatic restart |
| Code complexity | 500+ lines | 50 lines |

> **DLT = You define WHAT the data should look like. DLT handles HOW it gets there.**

---
## Chapter 2 — DLT Dataset Types

### Three Types of DLT Datasets

| Dataset Type | Storage | Refresh | Use Case |
|-------------|---------|---------|----------|
| **Streaming Table** | YES — stored as Delta | Incremental (new rows only) | Raw ingestion from files/Kafka |
| **Materialized View** | YES — stored as Delta | Full recompute every run | Aggregations, joins |
| **View** | NO — virtual only | Recomputed on every read | Reusable logic, temp transforms |

### When to use each

```
STREAMING TABLE:
  → Data arrives continuously (files landing in ADLS, Kafka topics)
  → Only new records are processed each run (incremental)
  → Replaces: readStream + writeStream in procedural code
  → Examples: Bronze layer ingestion

MATERIALIZED VIEW:
  → Results are stored physically (unlike a regular view)
  → Recomputed fully from scratch on each pipeline run
  → Works with GROUP BY, JOIN, aggregations
  → Examples: Silver cleansing, Gold aggregations

VIEW:
  → Just a named query — no storage, no Delta table
  → Used to break up complex logic into steps
  → Never shown in catalog — internal to the pipeline
  → Examples: Intermediate transformations between Bronze and Silver
```

### Exam tip

```
Q: Which dataset type stores aggregated results and refreshes fully each run?
A: Materialized View

Q: Which dataset type does NOT store data and is invisible in the catalog?
A: View

Q: Which dataset type processes only NEW rows each run?
A: Streaming Table
```

---
## Chapter 3 — DLT SQL Syntax

> **IMPORTANT:** DLT notebooks run INSIDE a DLT pipeline — not as regular notebooks.  
> To run these: create a DLT pipeline and attach this notebook as the source.

### Key SQL Keywords

```sql
-- Streaming Table (Bronze layer)
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT * FROM read_files('/raw/orders/', format => 'csv', header => 'true');

-- Materialized View (Silver / Gold)
CREATE OR REFRESH MATERIALIZED VIEW silver_orders
AS SELECT * FROM LIVE.bronze_orders WHERE OrderID IS NOT NULL;

-- View (intermediate step, no storage)
CREATE LIVE VIEW orders_with_channel
AS SELECT *, upper(OrderChannel) AS channel_upper FROM LIVE.bronze_orders;
```

### The LIVE Keyword

```
LIVE.bronze_orders  ← reads from another DLT table in this same pipeline

Without LIVE: Spark looks in catalog → may read stale or wrong version
With LIVE:    DLT resolves dependencies correctly → guaranteed fresh data

Rule: ALWAYS use LIVE.table_name to reference other DLT tables.
```

### read_files() — The DLT way to ingest files

```sql
-- read_files() = Auto Loader but in SQL
SELECT * FROM read_files(
    'abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/raw/',
    format => 'csv',
    header => 'true'
)
```

In [ ]:
# ─── DLT SQL — Full Medallion Pipeline ────────────────────────────────────────
# These SQL cells must run inside a DLT pipeline (attach to pipeline source)

# NOTE: Copy each SQL block into a separate SQL cell in your Databricks DLT notebook
# Do NOT run this Python cell — it is for display/teaching only

dlt_sql_bronze = """
-- ─── BRONZE: Raw ingestion from ADLS ─────────────────────────────────────────
CREATE OR REFRESH STREAMING TABLE bronze_orders
COMMENT 'Raw orders ingested from ADLS using Auto Loader'
AS SELECT
    *,
    current_timestamp() AS _ingested_at
FROM read_files(
    'abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/raw/',
    format      => 'csv',
    header      => 'true',
    inferSchema => 'true'
)
"""

dlt_sql_silver_view = """
-- ─── Intermediate VIEW (no storage) — normalise OrderChannel ─────────────────
CREATE LIVE VIEW orders_normalized
AS SELECT
    *,
    upper(trim(OrderChannel)) AS channel_clean
FROM LIVE.bronze_orders
"""

dlt_sql_silver = """
-- ─── SILVER: Cleansed, validated orders ──────────────────────────────────────
CREATE OR REFRESH MATERIALIZED VIEW silver_orders
COMMENT 'Cleansed orders — nulls removed, channel normalised'
AS SELECT
    OrderID, CustomerID,
    OrderDate, ShippingDate, ExpectedDeliveryDate, ActualDeliveryDate,
    channel_clean   AS OrderChannel,
    SupplierID, ShippingTierID,
    _ingested_at
FROM LIVE.orders_normalized
WHERE OrderID IS NOT NULL
  AND CustomerID IS NOT NULL
"""

dlt_sql_gold = """
-- ─── GOLD: Business aggregate — orders per channel ───────────────────────────
CREATE OR REFRESH MATERIALIZED VIEW gold_orders_by_channel
COMMENT 'Total orders grouped by order channel — for reporting'
AS SELECT
    OrderChannel,
    COUNT(*) AS order_count,
    MAX(OrderDate) AS latest_order_date
FROM LIVE.silver_orders
GROUP BY OrderChannel
"""

print("DLT SQL blocks ready — copy each into a separate SQL cell in your DLT notebook")
print()
print("bronze_orders          → STREAMING TABLE (Auto Loader from ADLS)")
print("orders_normalized      → LIVE VIEW (intermediate, no storage)")
print("silver_orders          → MATERIALIZED VIEW (cleaned)")
print("gold_orders_by_channel → MATERIALIZED VIEW (aggregated)")

---
## Chapter 4 — DLT Python Syntax

### Python Decorator Pattern

```python
import dlt
from pyspark.sql.functions import *

# Materialized View (default when using @dlt.table)
@dlt.table(name="gold_sales", comment="Monthly sales summary")
def gold_sales():
    return dlt.read("silver_orders").groupBy("order_channel").count()

# Streaming Table (incremental ingest)
@dlt.table
def bronze_orders():
    return spark.readStream.format("cloudFiles")...

# View (no storage)
@dlt.view
def orders_preview():
    return dlt.read("bronze_orders").limit(100)
```

### dlt.read() vs dlt.read_stream()

| Function | Type | Returns |
|----------|------|--------|
| `dlt.read("table")` | Batch | DataFrame (snapshot) |
| `dlt.read_stream("table")` | Streaming | StreamingDataFrame |

In [ ]:
# ─── DLT Python — Full Medallion Pipeline ─────────────────────────────────────
# These cells must be in a notebook attached to a DLT pipeline to run
# They cannot be run as a regular Databricks notebook (import dlt will fail)

import dlt
from pyspark.sql.functions import current_timestamp, upper, trim, col

storage_account = "amazonprojectadls"
container       = "amazon-data"
raw_path        = f"abfss://{container}@{storage_account}.dfs.core.windows.net/raw"


# ─── BRONZE: Streaming Table — ingest from ADLS via Auto Loader ───────────────
@dlt.table(
    name             = "bronze_orders",
    comment          = "Raw orders incrementally ingested from ADLS",
    table_properties = {"quality": "bronze"}
)
def bronze_orders():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "csv")
             .option("cloudFiles.schemaLocation", f"{raw_path}/_schemas/orders")
             .option("header", "true")
             .option("inferSchema", "true")
             .load(f"{raw_path}/")
             .withColumn("_ingested_at", current_timestamp())
    )


# ─── INTERMEDIATE VIEW: Normalise OrderChannel ────────────────────────────────
@dlt.view(name="orders_normalized")
def orders_normalized():
    return (
        dlt.read("bronze_orders")
           .withColumn("channel_clean", upper(trim(col("OrderChannel"))))
    )


# ─── SILVER: Materialized View — cleanse and validate ─────────────────────────
@dlt.table(
    name             = "silver_orders",
    comment          = "Validated and cleansed orders",
    table_properties = {"quality": "silver"}
)
@dlt.expect("non_null_order_id",  "OrderID IS NOT NULL")
@dlt.expect("non_null_customer",  "CustomerID IS NOT NULL")
def silver_orders():
    return (
        dlt.read("orders_normalized")
           .filter(col("OrderID").isNotNull())
           .filter(col("CustomerID").isNotNull())
           .select(
               "OrderID", "CustomerID", "OrderDate",
               "ShippingDate", "ExpectedDeliveryDate", "ActualDeliveryDate",
               "channel_clean", "SupplierID", "ShippingTierID", "_ingested_at"
           )
    )


# ─── GOLD: Materialized View — aggregate for reporting ────────────────────────
@dlt.table(
    name             = "gold_orders_by_channel",
    comment          = "Orders aggregated by channel for reporting",
    table_properties = {"quality": "gold"}
)
def gold_orders_by_channel():
    from pyspark.sql.functions import count, max as spark_max
    return (
        dlt.read("silver_orders")
           .groupBy("channel_clean")
           .agg(
               count("*").alias("order_count"),
               spark_max("OrderDate").alias("latest_order_date")
           )
    )

---
## Chapter 5 — Data Quality Expectations

### Three Modes of Data Quality

| Mode | SQL Syntax | Python Decorator | Behavior on Bad Row |
|------|-----------|-----------------|--------------------|
| **Warn** (default) | `CONSTRAINT name EXPECT (cond)` | `@dlt.expect("name", "cond")` | Row KEPT — warning logged in pipeline metrics |
| **Drop** | + `ON VIOLATION DROP ROW` | `@dlt.expect_or_drop("name", "cond")` | Row REMOVED from output |
| **Fail** | + `ON VIOLATION FAIL UPDATE` | `@dlt.expect_or_fail("name", "cond")` | Entire pipeline FAILS and stops |

### When to use each

```
WARN:  You care about quality but can't afford downtime. Monitor metric.
       "I want to know if nulls exist, but I still need the other rows."

DROP:  You need a clean Silver layer — bad rows must go.
       "Null order_id = incomplete record — reject it."

FAIL:  You CANNOT process anything if this rule breaks.
       "If no orders arrived today — something is broken upstream. Stop everything."
```

### Quick memory trick

```
EXPECT alone     = just warn (yellow flag, data stays)
EXPECT + DROP    = remove bad rows (red card)
EXPECT + FAIL    = stop the game (red card + end of match)
```

In [ ]:
# ─── DLT Data Quality — SQL Examples (for DLT SQL notebook cells) ─────────────

dlt_quality_sql = """
-- MODE 1: WARN — row kept, metric logged
CREATE OR REFRESH MATERIALIZED VIEW silver_orders_warn
CONSTRAINT valid_order_id EXPECT (OrderID IS NOT NULL)
AS SELECT * FROM LIVE.bronze_orders;


-- MODE 2: DROP — row removed silently
CREATE OR REFRESH MATERIALIZED VIEW silver_orders_clean
CONSTRAINT valid_order_id EXPECT (OrderID IS NOT NULL)
    ON VIOLATION DROP ROW
CONSTRAINT valid_customer EXPECT (CustomerID IS NOT NULL)
    ON VIOLATION DROP ROW
AS SELECT * FROM LIVE.bronze_orders;


-- MODE 3: FAIL — entire pipeline stops
CREATE OR REFRESH MATERIALIZED VIEW critical_orders
CONSTRAINT must_have_orders EXPECT (OrderID IS NOT NULL)
    ON VIOLATION FAIL UPDATE
AS SELECT * FROM LIVE.bronze_orders;


-- Multiple constraints on same table:
CREATE OR REFRESH MATERIALIZED VIEW silver_strict
CONSTRAINT valid_order     EXPECT (OrderID IS NOT NULL)     ON VIOLATION DROP ROW
CONSTRAINT valid_customer  EXPECT (CustomerID IS NOT NULL)  ON VIOLATION DROP ROW
CONSTRAINT valid_channel   EXPECT (OrderChannel IS NOT NULL)  -- WARN only
AS SELECT * FROM LIVE.bronze_orders;
"""

print("DLT Data Quality SQL — paste into DLT SQL cells when running as a pipeline")

In [ ]:
# ─── DLT Data Quality — Python Examples ───────────────────────────────────────
# These must run inside a DLT pipeline notebook

import dlt

# MODE 1: @dlt.expect — warn only, row kept
@dlt.table(name="silver_warn")
@dlt.expect("valid_order_id",  "OrderID IS NOT NULL")
@dlt.expect("valid_customer",  "CustomerID IS NOT NULL")
def silver_warn():
    return dlt.read("bronze_orders")
    # Bad rows KEPT — violation logged in pipeline event log


# MODE 2: @dlt.expect_or_drop — drop bad rows
@dlt.table(name="silver_clean")
@dlt.expect_or_drop("valid_order_id",  "OrderID IS NOT NULL")
@dlt.expect_or_drop("valid_customer",  "CustomerID IS NOT NULL")
def silver_clean():
    return dlt.read("bronze_orders")
    # Bad rows SILENTLY REMOVED — not visible in output table


# MODE 3: @dlt.expect_or_fail — fail the pipeline
@dlt.table(name="critical_orders")
@dlt.expect_or_fail("must_have_data", "OrderID IS NOT NULL")
def critical_orders():
    return dlt.read("bronze_orders")
    # If ANY row fails → pipeline STOPS with error


print("DLT Python data quality patterns ready")
print()
print("@dlt.expect          → warn only")
print("@dlt.expect_or_drop  → drop bad rows")
print("@dlt.expect_or_fail  → fail the pipeline")

---
## Chapter 6 — DLT Pipeline UI

### How to Create a DLT Pipeline

```
Databricks → Jobs & Pipelines → Pipelines → Create Pipeline

  Pipeline name   : GlobalMart Bronze-Gold Pipeline
  Product edition : Advanced  ← Required for data quality constraints
  Pipeline mode   : Triggered OR Continuous (explained in next chapter)
  Source code     : this notebook (attach it here)
  Target schema   : amazon_project.bronze (or silver/gold per layer)
  Storage path    : ADLS path OR Unity Catalog
  Cluster mode    : Auto (DLT manages the cluster automatically)
```

### Product Editions

| Edition | Features |
|---------|----------|
| Core | Basic pipeline execution only |
| Pro | Streaming Tables + Materialized Views |
| **Advanced** | Everything + **Data Quality Constraints** ← required for EXPECT |

> **Exam tip:** If the question mentions `EXPECT` or data quality — the answer is **Advanced edition**.

### The DAG — Lineage

```
After pipeline runs, DLT shows a visual DAG:

  bronze_orders
       │
       ├──→ orders_normalized (view)
       │          │
       │          └──→ silver_orders
       │                    │
       │                    └──→ gold_orders_by_channel

Each node shows:
  ✅ Green = passed
  🟡 Yellow = warnings (data quality violations, row kept)
  ❌ Red = failed
  Numbers on each node: records processed, warnings, errors
```

---
## Chapter 7 — Pipeline Modes × Environments

### 2 Modes × 2 Environments = 4 Combinations

| | Development | Production |
|-|-------------|------------|
| **Triggered** | Dev + Triggered | Prod + Triggered |
| **Continuous** | Dev + Continuous | Prod + Continuous |

---

### Triggered Mode — Process, Then Stop

```
Think of it like a washing machine:
  → You press START → it runs the full cycle → STOPS
  → It does NOT keep running waiting for more clothes

Triggered pipeline:
  → Processes all available new data
  → Stops the cluster when done
  → Scheduled to run again at specific time (e.g., every hour)
  → Cost-efficient: cluster only runs when needed
```

### Continuous Mode — Always Running

```
Think of it like a water tap:
  → Always open, water flows whenever you bring a glass
  → Does NOT wait for you to "start" — always processing

Continuous pipeline:
  → Cluster stays RUNNING 24/7
  → Processes new data as soon as it arrives (near real-time)
  → Higher cost — paying for idle cluster time
  → Required for true streaming use cases
```

---

### The KEY Rule for Dev vs Prod

```
Development mode:
  → Cluster STAYS RUNNING between runs
  → You click Run → pipeline executes → cluster stays live
  → Next Run is faster — cluster already warm
  → Great for iterating and testing code
  → CAUTION: costs money even when pipeline is idle

Production mode:
  → Cluster AUTO-TERMINATES after pipeline completes
  → Every run = fresh cluster startup (~5-10 min cold start)
  → Cost-efficient for production (no idle billing)
```

---

### The 4 Combinations

| Mode | Environment | Cluster Behavior | Use Case |
|------|------------|-----------------|----------|
| Triggered | Development | Stays running between runs | Build & test — fast iteration |
| Triggered | Production | Auto-terminates after each run | Scheduled ETL jobs (hourly/daily) |
| Continuous | Development | Stays running, processes stream | Test streaming pipelines |
| Continuous | Production | Auto-terminates only on manual stop | Live dashboards, real-time alerts |

> **The exam LOVES this question:** "In which mode does the cluster automatically terminate after the pipeline run?"  
> **Answer: Production mode (both Triggered and Continuous).**

---

### Exam Tips — Full DLT Summary

```
1. EXPECT alone = warn, row kept
   EXPECT + DROP ROW = row removed
   EXPECT + FAIL UPDATE = pipeline fails

2. Data quality in DLT requires Advanced edition.

3. Streaming Table: incremental, new rows only, Bronze layer.
   Materialized View: full recompute, stored, Silver/Gold.
   View: no storage, internal to pipeline, intermediate steps.

4. Use LIVE.table_name to reference DLT tables within same pipeline.

5. Production environment = cluster auto-terminates.
   Development environment = cluster stays running.

6. Triggered mode = scheduled/one-shot.
   Continuous mode = always on, near real-time.

7. DLT pipeline source = a notebook.
   DLT pipeline target = a catalog/schema (Unity Catalog).

8. In Python: @dlt.table = Materialized View by default.
              @dlt.view = View (no storage).

9. dlt.read() = batch. dlt.read_stream() = streaming.

10. The DAG visual in DLT UI shows: lineage + quality metrics per node.
```

In [ ]:
# ─── DLT Complete Pipeline — Python (all layers in one notebook) ───────────────
# Attach this notebook to a DLT pipeline to run it end-to-end

import dlt
from pyspark.sql.functions import current_timestamp, col, upper, trim, count, max as spark_max

STORAGE_ACCOUNT = "amazonprojectadls"
CONTAINER       = "amazon-data"
raw_path        = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/raw"


# ────────────────────────────────────────────────────────────
# BRONZE LAYER — Streaming Table — incremental Auto Loader ingest
# ────────────────────────────────────────────────────────────
@dlt.table(
    name             = "bronze_orders",
    comment          = "Raw orders from ADLS — incremental",
    table_properties = {"quality": "bronze", "delta.enableChangeDataFeed": "true"}
)
def bronze_orders():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "csv")
             .option("cloudFiles.schemaLocation", f"{raw_path}/_schemas/orders")
             .option("header", "true")
             .option("inferSchema", "true")
             .load(f"{raw_path}/")
             .withColumn("_ingested_at", current_timestamp())
    )


@dlt.table(
    name             = "bronze_customers",
    comment          = "Raw customers from ADLS — incremental",
    table_properties = {"quality": "bronze"}
)
def bronze_customers():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "csv")
             .option("cloudFiles.schemaLocation", f"{raw_path}/_schemas/customers")
             .option("header", "true")
             .option("inferSchema", "true")
             .load(f"{raw_path}/customers.csv")
             .withColumn("_ingested_at", current_timestamp())
    )


# ────────────────────────────────────────────────────────────
# SILVER LAYER — Materialized View — cleanse + validate
# ────────────────────────────────────────────────────────────
@dlt.table(
    name             = "silver_orders",
    comment          = "Cleansed orders with data quality enforced",
    table_properties = {"quality": "silver"}
)
@dlt.expect_or_drop("valid_order_id",  "OrderID IS NOT NULL")
@dlt.expect_or_drop("valid_customer",  "CustomerID IS NOT NULL")
@dlt.expect(         "has_channel",    "OrderChannel IS NOT NULL")  # warn only
def silver_orders():
    return (
        dlt.read("bronze_orders")
           .withColumn("channel_clean", upper(trim(col("OrderChannel"))))
    )


# ────────────────────────────────────────────────────────────
# GOLD LAYER — Materialized View — aggregated for reporting
# ────────────────────────────────────────────────────────────
@dlt.table(
    name             = "gold_orders_by_channel",
    comment          = "Orders aggregated by channel — business reporting",
    table_properties = {"quality": "gold"}
)
def gold_orders_by_channel():
    return (
        dlt.read("silver_orders")
           .groupBy("channel_clean")
           .agg(
               count("*").alias("order_count"),
               spark_max("OrderDate").alias("latest_order_date")
           )
    )


@dlt.table(
    name             = "gold_customer_activity",
    comment          = "Order count per customer — customer-level reporting",
    table_properties = {"quality": "gold"}
)
def gold_customer_activity():
    return (
        dlt.read("silver_orders")
           .groupBy("CustomerID")
           .agg(count("*").alias("order_count"))
           .filter(col("order_count") >= 1)
    )

---
## Chapter 8 — Practice Questions

```
Q1: A DLT pipeline needs to enforce that if any row has a null customer_id,
    the entire pipeline update should fail. Which syntax achieves this?

    A) CONSTRAINT val EXPECT (customer_id IS NOT NULL)
    B) CONSTRAINT val EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW
    C) CONSTRAINT val EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE  ← CORRECT
    D) @dlt.expect_or_drop("val", "customer_id IS NOT NULL")

────────────────────────────────────────────────────────────────────────────────

Q2: A DLT pipeline runs in Production mode. What happens to the cluster after
    the pipeline finishes?

    A) Cluster stays running to process the next batch
    B) Cluster automatically terminates  ← CORRECT
    C) Cluster pauses and waits
    D) Cluster restarts with new configuration

────────────────────────────────────────────────────────────────────────────────

Q3: Which DLT dataset type stores physically in Delta but recomputes
    completely with every pipeline run?

    A) Streaming Table
    B) View
    C) Materialized View  ← CORRECT
    D) Delta Live View

────────────────────────────────────────────────────────────────────────────────

Q4: A data engineer needs to run a DLT pipeline that processes data as soon
    as it arrives with minimum latency for live dashboards. Which mode?

    A) Triggered + Development
    B) Triggered + Production
    C) Continuous + Development
    D) Continuous + Production  ← CORRECT

────────────────────────────────────────────────────────────────────────────────

Q5: A DLT table references another DLT table in the same pipeline.
    Which syntax is correct?

    A) SELECT * FROM bronze_orders
    B) SELECT * FROM hive_metastore.bronze_orders
    C) SELECT * FROM LIVE.bronze_orders  ← CORRECT
    D) SELECT * FROM dlt.bronze_orders

────────────────────────────────────────────────────────────────────────────────

Q6: Which Databricks product edition is required to use
    data quality constraints in DLT?

    A) Core
    B) Pro
    C) Advanced  ← CORRECT
    D) Enterprise

────────────────────────────────────────────────────────────────────────────────

Q7: Which DLT dataset type has NO storage and is invisible in the catalog?

    A) Streaming Table
    B) Materialized View
    C) View  ← CORRECT
    D) Delta Table
```